In [4]:
import requests
import pandas as pd
import re
import time

# ==========================================
# THE PGA DIRECT-LINK DICTIONARY
# Bypasses search engines by pointing directly to the raw .txt files
# ==========================================
PGA_DIRECT_LINKS = {
    "1920s": [
        {
            "author": "F. Scott Fitzgerald",
            "works": [
                {"title": "The Great Gatsby", "url": "https://gutenberg.net.au/ebooks02/0200041.txt"},
                {"title": "This Side of Paradise", "url": "https://gutenberg.net.au/ebooks02/0200111.txt"},
                {"title": "The Beautiful and Damned", "url": "https://gutenberg.net.au/ebooks02/0200101.txt"},
                {"title": "Flappers and Philosophers", "url": "https://gutenberg.net.au/ebooks02/0200161.txt"},
                {"title": "Tales of the Jazz Age", "url": "https://gutenberg.net.au/ebooks02/0200121.txt"},
                {"title": "The Rich Boy", "url": "http://www.gutenberg.net.au/fsf/THE-RICH-BOY.txt"},
                {"title": "The Captured Shadow", "url": "https://gutenberg.net.au/fsf/THE-CAPTURED-SHADOW.txt"}
            ]
        },
        {
            "author": "Virginia Woolf",
            "works": [
                {"title": "Mrs Dalloway", "url": "https://gutenberg.net.au/ebooks02/0200991.txt"},
                {"title": "To the Lighthouse", "url": "https://gutenberg.net.au/ebooks01/0100101.txt"},
                {"title": "Orlando", "url": "https://gutenberg.net.au/ebooks02/0200331.txt"},
                {"title": "Jacob's Room", "url": "https://gutenberg.net.au/ebooks02/0200531.txt"},
                {"title": "Night and Day", "url": "https://gutenberg.net.au/ebooks15/1500341.txt"},
                {"title": "A Room of One's Own", "url": "https://gutenberg.net.au/ebooks02/0200791.txt"}
            ]
        },
        {
            "author": "Agatha Christie",
            "works": [
                {"title": "The Murder of Roger Ackroyd", "url": "https://gutenberg.net.au/ebooks08/0800361.txt"}
            ]
        },
        {
            "author": "Ernest Hemingway",
            "works": [
                {"title": "A Farewell to Arms", "url": "https://gutenberg.net.au/ebooks06/0600011.txt"}
            ]
        }
    ],
    "1930s": [
        {
            "author": "George Orwell",
            "works": [
                {"title": "Down and Out in Paris and London", "url": "https://gutenberg.net.au/ebooks01/0100171.txt"},
                {"title": "Burmese Days", "url": "https://gutenberg.net.au/ebooks02/0200051.txt"},
                {"title": "A Clergyman's Daughter", "url": "https://gutenberg.net.au/ebooks02/0200061.txt"},
                {"title": "Keep the Aspidistra Flying", "url": "https://gutenberg.net.au/ebooks02/0200141.txt"},
                {"title": "The Road to Wigan Pier", "url": "https://gutenberg.net.au/ebooks02/0200391.txt"}
            ]
        },
        {
            "author": "Aldous Huxley",
            "works": [
                {"title": "Brave New World", "url": "https://gutenberg.net.au/ebooks03/0300051.txt"}
            ]
        },
        {
            "author": "Virginia Woolf",
            "works": [
                {"title": "The Waves", "url": "https://gutenberg.net.au/ebooks02/0201091.txt"}
            ]
        },
        {
            "author": "Agatha Christie",
            "works": [
                {"title": "Murder on the Orient Express", "url": "https://gutenberg.net.au/ebooks08/0800041.txt"}
            ]
        },
        {
            "author": "H.G. Wells",
            "works": [
                {"title": "The Shape of Things to Come", "url": "https://gutenberg.net.au/ebooks03/0301391.txt"}
            ]
        },
        {
            "author": "Sinclair Lewis",
            "works": [
                {"title": "It Can't Happen Here", "url": "https://gutenberg.net.au/ebooks03/0301001.txt"}
            ]
        }
    ],
    "1940s": [
        {
            "author": "George Orwell",
            "works": [
                {"title": "Nineteen Eighty-Four", "url": "https://gutenberg.net.au/ebooks01/0100021.txt"},
                {"title": "Animal Farm", "url": "https://gutenberg.net.au/ebooks01/0100011.txt"},
                {"title": "Homage to Catalonia", "url": "https://gutenberg.net.au/ebooks02/0201111.txt"},
                {"title": "Coming Up for Air", "url": "https://gutenberg.net.au/ebooks02/0200211.txt"}
            ]
        },
        {
            "author": "Arthur Koestler",
            "works": [
                {"title": "Darkness at Noon", "url": "https://gutenberg.net.au/ebooks01/0100031.txt"}
            ]
        },
        {
            "author": "Graham Greene",
            "works": [
                {"title": "The Heart of the Matter", "url": "https://gutenberg.net.au/ebooks06/0607141.txt"}
            ]
        }
    ],
    "1950s": [
        {
            "author": "Ian Fleming",
            "works": [
                {"title": "Casino Royale", "url": "https://gutenberg.net.au/ebooks05/0500461.txt"},
                {"title": "Goldfinger", "url": "https://gutenberg.net.au/ebooks05/0500511.txt"}
            ]
        },
        {
            "author": "Ernest Hemingway",
            "works": [
                {"title": "The Old Man and the Sea", "url": "https://gutenberg.net.au/ebooks06/0600041.txt"}
            ]
        },
        {
            "author": "Arthur C. Clarke",
            "works": [
                {"title": "Childhood's End", "url": "https://gutenberg.net.au/ebooks06/0600141.txt"}
            ]
        },
        {
            "author": "John Wyndham",
            "works": [
                {"title": "The Day of the Triffids", "url": "https://gutenberg.net.au/ebooks01/0100211.txt"}
            ]
        }
    ],
    "1960s": [
        {
            "author": "Arthur C. Clarke",
            "works": [
                {"title": "2001: A Space Odyssey", "url": "https://gutenberg.net.au/ebooks06/0600151.txt"}
            ]
        },
        {
            "author": "John le Carré",
            "works": [
                {"title": "The Spy Who Came in from the Cold", "url": "https://gutenberg.net.au/ebooks06/0600061.txt"}
            ]
        },
        {
            "author": "Ian Fleming",
            "works": [
                {"title": "You Only Live Twice", "url": "https://gutenberg.net.au/ebooks05/0500551.txt"}
            ]
        }
    ]
}
def extract_massive_midcentury():
    results = []
    MAX_PARAGRAPHS = 60 
    
    print("\n" + "="*70)
    print("🚀 INITIALIZING BULLETPROOF DIRECT-LINK MINER (1920-1960)")
    print("="*70)
    
    # Simple header, no advanced spoofing needed for raw text files
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    for decade, authors in PGA_DIRECT_LINKS.items():
        print(f"\n📂 Processing {decade}...")
        
        for author_data in authors:
            author = author_data["author"]
            
            for work in author_data["works"]:
                title = work["title"]
                url = work["url"]
                
                # Skip if URL hasn't been filled in yet
                if url == "PASTE_URL_HERE":
                    continue
                    
                print(f"  📥 Downloading: {title} by {author}...")
                
                try:
                    response = requests.get(url, headers=headers, timeout=15)
                    if response.status_code == 200:
                        text = response.text.replace('\r', '')
                        
                        # Clean PGA Boilerplate
                        if "Language:" in text:
                            text = text.split("Language:", 1)[-1]
                        if "THE END" in text:
                            text = text.split("THE END")[0]
                        
                        count = 0
                        for p in text.split('\n\n'):
                            clean_p = re.sub(r'\s+', ' ', p).strip()
                            
                            if 150 < len(clean_p) < 1000:
                                results.append({
                                    "author": author,
                                    "title": title,
                                    "time_period": int(decade[:4]),
                                    "language": "en",
                                    "source": "pga_direct",
                                    "text": clean_p
                                })
                                count += 1
                                if count >= MAX_PARAGRAPHS:
                                    break
                                    
                        print(f"     ✅ Extracted {count} rows.")
                    else:
                        print(f"     ❌ Server returned status {response.status_code}")
                    
                    time.sleep(1) # Be polite to the Australian server
                    
                except Exception as e:
                    print(f"     ⚠️ Network Error: {e}")

    # ==========================================
    # 3. EXPORT
    # ==========================================
    df = pd.DataFrame(results)
    
    if not df.empty:
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        output_file = "massive_midcentury_english.csv"
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
        
        print("\n" + "="*70)
        print(f"🎉 SUCCESS! Saved {len(df)} rows to {output_file}")
        print("="*70)
    else:
        print("\n⚠️ No data was extracted. Ensure URLs are correctly formatted.")

if __name__ == "__main__":
    extract_massive_midcentury()


🚀 INITIALIZING BULLETPROOF DIRECT-LINK MINER (1920-1960)

📂 Processing 1920s...
  📥 Downloading: The Great Gatsby by F. Scott Fitzgerald...
     ✅ Extracted 60 rows.
  📥 Downloading: This Side of Paradise by F. Scott Fitzgerald...
     ✅ Extracted 60 rows.
  📥 Downloading: The Beautiful and Damned by F. Scott Fitzgerald...
     ✅ Extracted 60 rows.
  📥 Downloading: Flappers and Philosophers by F. Scott Fitzgerald...
     ✅ Extracted 60 rows.
  📥 Downloading: Tales of the Jazz Age by F. Scott Fitzgerald...
     ✅ Extracted 60 rows.
  📥 Downloading: The Rich Boy by F. Scott Fitzgerald...
     ✅ Extracted 60 rows.
  📥 Downloading: The Captured Shadow by F. Scott Fitzgerald...
     ✅ Extracted 60 rows.
  📥 Downloading: Mrs Dalloway by Virginia Woolf...
     ✅ Extracted 60 rows.
  📥 Downloading: To the Lighthouse by Virginia Woolf...
     ✅ Extracted 60 rows.
  📥 Downloading: Orlando by Virginia Woolf...
     ✅ Extracted 60 rows.
  📥 Downloading: Jacob's Room by Virginia Woolf...
     ✅ Ex